In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


In [2]:
val_binary = pd.read_csv("Extraction/val_binary.tsv", sep="\t")
test_binary = pd.read_csv("Extraction/test_binary.tsv", sep="\t")
metadata_cols = ["Entry", "Length", "Sequence"]
concept_cols = [c for c in val_binary.columns if c not in metadata_cols]

In [3]:
def calculate_f1_array(precision, recall):
    denom = precision + recall
    return np.divide(
        2 * precision * recall,
        denom,
        out=np.zeros_like(denom, dtype=float),
        where=denom > 0
    )

def compare_features_to_concepts_fast(
    A,
    binary_df,
    concept_cols,
    thresholds=(0, 0.15, 0.5, 0.6, 0.8),
):
    """
    A: normalized activations, shape [n_proteins, n_features]
    binary_df: val_binary/test_binary
    concept_cols: concept label columns

    Returns dataframe with:
    concept, feature, threshold, precision, recall, f1
    """

    A = np.asarray(A)
    Y = binary_df[concept_cols].values.astype(bool)

    n_proteins, n_features = A.shape
    n_concepts = Y.shape[1]

    positives = Y.sum(axis=0)  # [n_concepts]
    results = []

    for threshold in thresholds:
        A_bin = A > threshold  # [n_proteins, n_features]

        # Matrix multiplication gives TP:
        # Y.T: [n_concepts, n_proteins]
        # A_bin: [n_proteins, n_features]
        # tp: [n_concepts, n_features]
        tp = Y.T.astype(np.int32) @ A_bin.astype(np.int32)

        pred_pos = A_bin.sum(axis=0)  # [n_features]
        fp = pred_pos[None, :] - tp

        precision = np.divide(
            tp,
            tp + fp,
            out=np.zeros_like(tp, dtype=float),
            where=(tp + fp) > 0,
        )

        recall = np.divide(
            tp,
            positives[:, None],
            out=np.zeros_like(tp, dtype=float),
            where=positives[:, None] > 0,
        )

        f1 = calculate_f1_array(precision, recall)

        concept_idx, feature_idx = np.nonzero(tp > 0)

        df_t = pd.DataFrame({
            "concept": [concept_cols[i] for i in concept_idx],
            "feature": feature_idx,
            "threshold": threshold,
            "precision": precision[concept_idx, feature_idx],
            "recall": recall[concept_idx, feature_idx],
            "f1": f1[concept_idx, feature_idx],
            #"tp": tp[concept_idx, feature_idx],
            #"fp": fp[concept_idx, feature_idx],
            #"positive_labels": positives[concept_idx],
        })

        results.append(df_t)

    return pd.concat(results, ignore_index=True)

CLS

In [8]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_8/embeddings_cls_sae_8_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_8/embeddings_cls_sae_8_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 8)
(25000, 8)
val f1: 0.07123
test f1: 0.06667
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,0,0.5,0.456291,0.575575,0.509038,0.449196,0.565307,0.500607


In [9]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_16/embeddings_cls_sae_16_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_16/embeddings_cls_sae_16_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 16)
(25000, 16)
val f1: 0.0745
test f1: 0.06933
Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [10]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_32/embeddings_cls_sae_32_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_32/embeddings_cls_sae_32_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 32)
(25000, 32)
val f1: 0.08711
test f1: 0.08029
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,29,0.15,0.355037,0.885687,0.506884,0.354455,0.887318,0.506557


In [11]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_64/embeddings_cls_sae_64_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_64/embeddings_cls_sae_64_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 64)
(25000, 64)
val f1: 0.09987
test f1: 0.09028
Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [12]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_128/embeddings_cls_sae_128_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_128/embeddings_cls_sae_128_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 128)
(25000, 128)
val f1: 0.10354
test f1: 0.09334
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 1
Survival rate: 0.5


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,79,0.15,0.384928,0.758869,0.510773,0.383036,0.758054,0.50892


In [13]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_320/embeddings_cls_sae_320_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_320/embeddings_cls_sae_320_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 320)
(25000, 320)
val f1: 0.11974
test f1: 0.10647
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 0
Survival rate: 0.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [14]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_2560/embeddings_cls_sae_2560_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_2560/embeddings_cls_sae_2560_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 2560)
(25000, 2560)
val f1: 0.14753
test f1: 0.13049
Validation pairs with F1 > 0.5: 7
Those also with test F1 > 0.5: 5
Survival rate: 0.7142857142857143


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,1745,0.15,0.444191,0.794074,0.569701,0.447384,0.799783,0.573797
1,GO:0106310,1436,0.50,0.667870,0.460199,0.544919,0.674377,0.446408,0.537208
4,GO:0005737,182,0.00,0.357613,0.879766,0.508520,0.361507,0.878466,0.512223
2,transmembrane,1111,0.15,0.375823,0.794572,0.510286,0.373417,0.803737,0.509923
3,protein kinase,1436,0.50,0.720217,0.395050,0.510230,0.741993,0.380822,0.503319


In [15]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_5120/embeddings_cls_sae_5120_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_5120/embeddings_cls_sae_5120_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 5120)
(25000, 5120)
val f1: 0.15412
test f1: 0.13505
Validation pairs with F1 > 0.5: 6
Those also with test F1 > 0.5: 5
Survival rate: 0.8333333333333334


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,2269,0.15,0.475602,0.748539,0.581643,0.477192,0.749354,0.583078
1,GO:0106310,3073,0.50,0.703985,0.461443,0.557476,0.721905,0.446408,0.551674
4,GO:0005737,1783,0.00,0.375979,0.823938,0.516341,0.381343,0.819246,0.520434
3,transmembrane,554,0.15,0.424782,0.684816,0.524329,0.420934,0.670723,0.517250
5,protein kinase,4601,0.50,0.687291,0.406931,0.511194,0.699675,0.393607,0.503799


Layer mean

In [16]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_8/embeddings_layer_mean_sae_8_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_8/embeddings_layer_mean_sae_8_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 8)
(25000, 8)
val f1: 0.07012
test f1: 0.06715
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,2,0.15,0.378177,0.827239,0.519062,0.37749,0.821531,0.517289


In [23]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_16/embeddings_layer_mean_sae_16_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_16/embeddings_layer_mean_sae_16_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 16)
(25000, 16)
val f1: 0.08291
test f1: 0.07897
Validation pairs with F1 > 0.5: 5
Those also with test F1 > 0.5: 3
Survival rate: 0.6


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
1,transmembrane,12,0.15,0.494726,0.741154,0.593372,0.492496,0.731231,0.588576
0,GO:0005506,0,0.60,0.729825,0.506083,0.597701,0.699620,0.494624,0.579528
2,heme,0,0.60,0.719298,0.488095,0.581560,0.676806,0.463542,0.550232


In [17]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_32/embeddings_layer_mean_sae_32_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_32/embeddings_layer_mean_sae_32_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 32)
(25000, 32)
val f1: 0.10744
test f1: 0.10144
Validation pairs with F1 > 0.5: 10
Those also with test F1 > 0.5: 7
Survival rate: 0.7


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
1,GO:0005506,1,0.80,0.975000,0.474453,0.638298,0.937853,0.446237,0.604736
0,GO:0003925,5,0.50,0.523416,0.867580,0.652921,0.455657,0.818681,0.585462
2,heme,1,0.80,0.965000,0.459524,0.622581,0.903955,0.416667,0.570410
3,transmembrane,12,0.15,0.569064,0.584507,0.576682,0.558005,0.563936,0.560955
4,GO:0003924,5,0.50,0.699725,0.463504,0.557629,0.675841,0.421756,0.519389
6,1.14,1,0.80,0.805000,0.397531,0.532231,0.790960,0.371353,0.505415
8,GO:0007186,25,0.60,0.557803,0.461722,0.505236,0.559420,0.457346,0.503259


In [18]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_64/embeddings_layer_mean_sae_64_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_64/embeddings_layer_mean_sae_64_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 64)
(25000, 64)
val f1: 0.12414
test f1: 0.11549
Validation pairs with F1 > 0.5: 9
Those also with test F1 > 0.5: 9
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
1,pyridoxal 5'-phosphate,28,0.60,0.765060,0.557018,0.644670,0.823529,0.590717,0.687961
0,GO:0005506,5,0.80,0.975962,0.493917,0.655897,0.942408,0.483871,0.639432
3,heme,5,0.80,0.966346,0.478571,0.640127,0.910995,0.453125,0.605217
2,GO:0003925,1,0.60,0.506631,0.872146,0.640940,0.449848,0.813187,0.579256
4,transmembrane,10,0.15,0.445803,0.795431,0.571376,0.439119,0.793109,0.565268
5,1.14,5,0.80,0.802885,0.412346,0.544861,0.806283,0.408488,0.542254
7,GO:0020037,56,0.60,0.920354,0.382353,0.540260,0.929293,0.355212,0.513966
8,GO:0005634,15,0.15,0.424847,0.651217,0.514221,0.425374,0.645780,0.512901
6,GO:0003924,1,0.60,0.665782,0.458029,0.542703,0.656535,0.412214,0.506448


In [19]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_128/embeddings_layer_mean_sae_128_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_128/embeddings_layer_mean_sae_128_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 128)
(25000, 128)
val f1: 0.13747
test f1: 0.12717
Validation pairs with F1 > 0.5: 12
Those also with test F1 > 0.5: 8
Survival rate: 0.6666666666666666


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005506,73,0.80,1.000000,0.506083,0.672052,1.000000,0.491935,0.659459
4,transmembrane,127,0.15,0.600935,0.684129,0.639839,0.591106,0.676723,0.631024
2,heme,73,0.80,0.985577,0.488095,0.652866,0.967213,0.460938,0.624339
1,pyridoxal 5'-phosphate,58,0.60,0.685990,0.622807,0.652874,0.668293,0.578059,0.619910
3,GO:0003925,5,0.80,0.561056,0.776256,0.651341,0.477778,0.708791,0.570796
6,1.14,73,0.80,0.822115,0.422222,0.557912,0.846995,0.411141,0.553571
7,GO:0020037,73,0.80,1.000000,0.382353,0.553191,1.000000,0.353282,0.522111
9,GO:0007186,61,0.50,0.526718,0.495215,0.510481,0.531746,0.476303,0.502500


In [20]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_320/embeddings_layer_mean_sae_320_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_320/embeddings_layer_mean_sae_320_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 320)
(25000, 320)
val f1: 0.15467
test f1: 0.14497
Validation pairs with F1 > 0.5: 14
Those also with test F1 > 0.5: 12
Survival rate: 0.8571428571428571


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,6.2,254,0.80,0.905660,0.615385,0.732824,0.779661,0.676471,0.724409
1,GO:0005506,53,0.60,0.920354,0.506083,0.653061,0.934010,0.494624,0.646749
4,transmembrane,269,0.15,0.571668,0.704225,0.631060,0.565638,0.691292,0.622185
2,heme,53,0.60,0.907080,0.488095,0.634675,0.903553,0.463542,0.612737
5,protein kinase,60,0.50,0.609950,0.606931,0.608437,0.597056,0.592694,0.594867
3,ig-like,161,0.50,0.689751,0.587264,0.634395,0.647590,0.523114,0.578735
7,GO:0106310,60,0.50,0.517413,0.646766,0.574903,0.509660,0.652532,0.572314
9,1.14,16,0.80,0.821782,0.409877,0.546952,0.839779,0.403183,0.544803
8,GO:0004252,106,0.50,0.604839,0.508475,0.552486,0.566038,0.490196,0.525394
10,GO:0020037,53,0.60,0.920354,0.382353,0.540260,0.934010,0.355212,0.514685


In [21]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_2560/embeddings_layer_mean_sae_2560_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_2560/embeddings_layer_mean_sae_2560_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 2560)
(25000, 2560)
val f1: 0.19674
test f1: 0.18362
Validation pairs with F1 > 0.5: 30
Those also with test F1 > 0.5: 28
Survival rate: 0.9333333333333333


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,cadherin,957,0.8,0.978723,0.779661,0.867925,1.000000,0.807692,0.893617
3,protein kinase,447,0.5,0.736939,0.796040,0.765350,0.754223,0.815525,0.783677
2,abc transporter,250,0.6,0.803922,0.732143,0.766355,0.796875,0.629630,0.703448
1,6.2,314,0.6,0.768293,0.807692,0.787500,0.645570,0.750000,0.693878
4,ig-like,1388,0.6,0.842767,0.632075,0.722372,0.784127,0.600973,0.680441
7,GO:0106310,447,0.5,0.571952,0.776119,0.658575,0.570946,0.796231,0.665027
10,fad,371,0.5,0.851163,0.515493,0.642105,0.846847,0.526611,0.649396
5,GO:0004252,581,0.5,0.714286,0.644068,0.677362,0.695489,0.604575,0.646853
6,GO:0005506,431,0.8,1.000000,0.501217,0.667747,1.000000,0.467742,0.637363
11,transmembrane,2404,0.0,0.723531,0.564583,0.634250,0.726369,0.563764,0.634820


In [22]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_5120/embeddings_layer_mean_sae_5120_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_5120/embeddings_layer_mean_sae_5120_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 5120)
(25000, 5120)
val f1: 0.20273
test f1: 0.19115
Validation pairs with F1 > 0.5: 30
Those also with test F1 > 0.5: 26
Survival rate: 0.8666666666666667


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,cadherin,2747,0.80,0.958333,0.779661,0.859813,0.851064,0.769231,0.808081
6,c-type lectin,259,0.60,0.793651,0.581395,0.671141,0.816901,0.698795,0.753247
2,pyridoxal 5'-phosphate,3889,0.60,0.875000,0.614035,0.721649,0.894410,0.607595,0.723618
5,protein kinase,3294,0.50,0.669894,0.685149,0.677435,0.675983,0.706849,0.691071
1,GO:0003925,2172,0.80,0.728111,0.721461,0.724771,0.681319,0.681319,0.681319
13,krab,2985,0.60,0.539683,0.653846,0.591304,0.709091,0.639344,0.672414
4,ig-like,4336,0.50,0.645228,0.733491,0.686534,0.621444,0.690998,0.654378
7,GO:0005506,2146,0.80,1.000000,0.501217,0.667747,0.988889,0.478495,0.644928
3,6.2,3631,0.50,0.605769,0.807692,0.692308,0.530612,0.764706,0.626506
24,abc transporter,2425,0.60,0.535714,0.535714,0.535714,0.714286,0.555556,0.625000
